# SE-Net (Squeeze-and-Excitation)

Hu, Shen, Sun, *Squeeze-and-Excitation Networks*, CVPR 2018 ([arXiv:1709.01507](https://arxiv.org/abs/1709.01507)).

An SE block reweights a feature map's channels using a descriptor pooled from the *whole* image (squeeze), a small bottleneck MLP (excitation), then rescales the original feature map (see `model.py` and `papers/README.md`). This notebook trains the same small residual backbone **with and without** SE blocks on real CIFAR-10, so SE's effect is a direct ablation, not just asserted.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from cnn_playground.data import load_cifar10
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import SEResNetModel

set_seed(0)
# device options: 'auto' (default, picks cuda/mps if available), 'cpu', 'cuda', 'mps'
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_ds = load_cifar10(train=True)
test_ds = load_cifar10(train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.shape[0]
    return correct / total

def train(model, epochs=20, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'test_acc': []}
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item() * x.shape[0]
        history['train_loss'].append(running / len(train_ds))
        history['test_acc'].append(evaluate(model, test_loader))
    return history

In [ ]:
set_seed(0)
plain_model = SEResNetModel(use_se=False).to(device)
plain_history = train(plain_model)
print(f"plain final test accuracy: {plain_history['test_acc'][-1]:.3f}")

In [ ]:
set_seed(0)
se_model = SEResNetModel(use_se=True).to(device)
se_history = train(se_model)
print(f"SE-augmented final test accuracy: {se_history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(plain_history['train_loss'], label='plain'); axes[0].plot(se_history['train_loss'], label='SE'); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(plain_history['test_acc'], label='plain'); axes[1].plot(se_history['test_acc'], label='SE'); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
fig.tight_layout()
plt.show()